# AMR-UnderDifferentNoises-DL: Sonuç Analizi ve Karşılaştırma

Bu notebook, fine-tuning işlemi sonrasında elde edilen `.pkl` uzantılı sonuç dosyalarını (Accuracy, F1 Score, MCC) Google Drive'dan okur ve **modeller arası ile veri setleri arası kıyaslamaları** tek bir grafik üzerinde birleştirir.
Ayrıca oluşturulan bu birleşik grafikleri yüksek çözünürlüklü olarak Drive'ınıza kaydeder.

In [ ]:
# 1. Drive'a Bağlan ve Gerekli Kütüphaneleri Yükle
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = '/content/drive/MyDrive/AMR-Project/fine_tuning_results'
if not os.path.exists(RESULTS_DIR):
    RESULTS_DIR = '/content/drive/MyDrive/AMR-Project/results'

SAVE_DIR = os.path.join(RESULTS_DIR, 'Karsilastirma_Grafikleri')
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"\nSonuçların okunacağı dizin: {RESULTS_DIR}")
print(f"Grafiklerin otomatik kaydedileceği dizin: {SAVE_DIR}")


In [ ]:
# 2. Verileri Okuma Fonksiyonu
def load_pkl_data(model, dataset, metric_file='acc.pkl'):
    folder_name = f"{model}_{dataset}"
    file_path = os.path.join(RESULTS_DIR, folder_name, 'predictions', metric_file)
    if not os.path.exists(file_path):
        file_path = os.path.join(RESULTS_DIR, folder_name, metric_file)
    if os.path.exists(file_path):
        with open(file_path, 'rb') as f:
            return pickle.load(f)
    else:
        return None

models = ['mcldnn', 'petcgdnn']
datasets = ['rayleigh', 'rician_k3', 'rician_k10']


## A) CLASSIFICATION ACCURACY (Doğruluk Oranı) Analizleri

In [ ]:
# 3. Model Karşılaştırması (ACCURACY)
def plot_model_comparison_acc(dataset_name, dataset_title):
    mcldnn_acc = load_pkl_data('mcldnn', dataset_name, 'acc.pkl')
    petcgdnn_acc = load_pkl_data('petcgdnn', dataset_name, 'acc.pkl')
    if mcldnn_acc is None or petcgdnn_acc is None: return
        
    snrs_sorted = sorted(list(mcldnn_acc.keys()))
    mcl_vals = [mcldnn_acc[s]*100 for s in snrs_sorted]
    pet_vals = [petcgdnn_acc[s]*100 for s in snrs_sorted]
    
    plt.figure(figsize=(10, 6), dpi=150)
    plt.plot(snrs_sorted, mcl_vals, 'o-', linewidth=2.5, label='MCLDNN', color='#E63946')
    plt.plot(snrs_sorted, pet_vals, 's-', linewidth=2.5, label='PET-CGDNN', color='#1D3557')
    plt.xlabel("Signal to Noise Ratio (SNR) in dB", fontsize=12)
    plt.ylabel("Classification Accuracy (%)", fontsize=12)
    plt.title(f"Model Accuracy Comparison on {dataset_title} Channel", fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([0, 105])
    plt.savefig(os.path.join(SAVE_DIR, f"model_comp_accuracy_{dataset_name}.png"), bbox_inches='tight')
    plt.show()

print("========== MODEL KARŞILAŞTIRMALARI (ACCURACY) ==========")
plot_model_comparison_acc('rayleigh', 'Rayleigh Fading')
plot_model_comparison_acc('rician_k3', 'Rician Fading (K=3)')
plot_model_comparison_acc('rician_k10', 'Rician Fading (K=10)')


In [ ]:
# 4. Kanal Karşılaştırması (ACCURACY)
def plot_channel_comparison_acc(model_name, model_title):
    rayleigh_acc = load_pkl_data(model_name, 'rayleigh', 'acc.pkl')
    rician3_acc = load_pkl_data(model_name, 'rician_k3', 'acc.pkl')
    rician10_acc = load_pkl_data(model_name, 'rician_k10', 'acc.pkl')
    if rayleigh_acc is None or rician3_acc is None or rician10_acc is None: return
        
    snrs_sorted = sorted(list(rayleigh_acc.keys()))
    plt.figure(figsize=(10, 6), dpi=150)
    plt.plot(snrs_sorted, [rayleigh_acc[s]*100 for s in snrs_sorted], 'o-', linewidth=2.5, label='Rayleigh', color='#E63946')
    plt.plot(snrs_sorted, [rician3_acc[s]*100 for s in snrs_sorted], 's-', linewidth=2.5, label='Rician K=3', color='#2A9D8F')
    plt.plot(snrs_sorted, [rician10_acc[s]*100 for s in snrs_sorted], '^-', linewidth=2.5, label='Rician K=10', color='#457B9D')
    plt.xlabel("SNR (dB)", fontsize=12)
    plt.ylabel("Classification Accuracy (%)", fontsize=12)
    plt.title(f"Channel Fading Impact on {model_title} (Accuracy)", fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([0, 105])
    plt.savefig(os.path.join(SAVE_DIR, f"channel_comp_accuracy_{model_name}.png"), bbox_inches='tight')
    plt.show()

print("========== KANAL KARŞILAŞTIRMALARI (ACCURACY) ==========")
plot_channel_comparison_acc('mcldnn', 'MCLDNN')
plot_channel_comparison_acc('petcgdnn', 'PET-CGDNN')


## B) F1 SCORE (Macro) Analizleri

In [ ]:
# 5. Model Karşılaştırması (F1 SCORE)
def plot_model_comparison_f1(dataset_name, dataset_title):
    mcl_f1 = load_pkl_data('mcldnn', dataset_name, 'f1_macro_scores.pkl')
    pet_f1 = load_pkl_data('petcgdnn', dataset_name, 'f1_macro_scores.pkl')
    if mcl_f1 is None or pet_f1 is None: return
        
    snrs_sorted = sorted(list(mcl_f1.keys()))
    plt.figure(figsize=(10, 6), dpi=150)
    plt.plot(snrs_sorted, [mcl_f1[s] for s in snrs_sorted], 'o-', linewidth=2.5, label='MCLDNN', color='#E63946')
    plt.plot(snrs_sorted, [pet_f1[s] for s in snrs_sorted], 's-', linewidth=2.5, label='PET-CGDNN', color='#1D3557')
    plt.xlabel("SNR (dB)", fontsize=12)
    plt.ylabel("F1 Score (Macro)", fontsize=12)
    plt.title(f"Model F1 Score Comparison on {dataset_title} Channel", fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([0, 1.05])
    plt.savefig(os.path.join(SAVE_DIR, f"model_comp_f1_{dataset_name}.png"), bbox_inches='tight')
    plt.show()

print("========== MODEL KARŞILAŞTIRMALARI (F1 SCORE) ==========")
plot_model_comparison_f1('rayleigh', 'Rayleigh Fading')
plot_model_comparison_f1('rician_k3', 'Rician Fading (K=3)')
plot_model_comparison_f1('rician_k10', 'Rician Fading (K=10)')


## C) MCC (Matthews Correlation Coefficient) Analizleri

In [ ]:
# 6. Model Karşılaştırması (MCC)
def plot_model_comparison_mcc(dataset_name, dataset_title):
    mcl_mcc = load_pkl_data('mcldnn', dataset_name, 'mcc_metric_scores.pkl')
    pet_mcc = load_pkl_data('petcgdnn', dataset_name, 'mcc_metric_scores.pkl')
    if mcl_mcc is None or pet_mcc is None: return
        
    snrs_sorted = sorted(list(mcl_mcc.keys()))
    plt.figure(figsize=(10, 6), dpi=150)
    plt.plot(snrs_sorted, [mcl_mcc[s] for s in snrs_sorted], 'o-', linewidth=2.5, label='MCLDNN', color='#E63946')
    plt.plot(snrs_sorted, [pet_mcc[s] for s in snrs_sorted], 's-', linewidth=2.5, label='PET-CGDNN', color='#1D3557')
    plt.xlabel("SNR (dB)", fontsize=12)
    plt.ylabel("MCC", fontsize=12)
    plt.title(f"Model MCC Comparison on {dataset_title} Channel", fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([-0.1, 1.05])
    plt.savefig(os.path.join(SAVE_DIR, f"model_comp_mcc_{dataset_name}.png"), bbox_inches='tight')
    plt.show()

print("========== MODEL KARŞILAŞTIRMALARI (MCC) ==========")
plot_model_comparison_mcc('rayleigh', 'Rayleigh Fading')
plot_model_comparison_mcc('rician_k3', 'Rician Fading (K=3)')
plot_model_comparison_mcc('rician_k10', 'Rician Fading (K=10)')


In [ ]:
# 7. İstatistiksel Karşılaştırma Raporu (Yazılı Çıktı)
print("========== YAZILI PERFORMANS ÖZET RAPORU ==========\n")
for dataset in datasets:
    print(f"[{dataset.upper()}] İÇİN GENEL ÖZET:")
    for model in models:
        acc_data = load_pkl_data(model, dataset, 'acc.pkl')
        f1_data = load_pkl_data(model, dataset, 'f1_macro_scores.pkl')
        if acc_data and f1_data:
            avg_acc = np.mean(list(acc_data.values())) * 100
            max_acc = np.max(list(acc_data.values())) * 100
            avg_f1 = np.mean(list(f1_data.values()))
            print(f"  - {model.upper():<8} : Ort. Acc= %{avg_acc:.2f} | Max Acc= %{max_acc:.2f} | Ort. F1= {avg_f1:.4f}")
    print("-" * 60)
